In [9]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [10]:
#session 

spark = (( SparkSession
            .builder
                .appName('statecities')
                .config('spark.jars', f'{sqlserverjar}')
                .config('spark.driver.extraClassPath', f'{sqlserverjar}')
                .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
        ))

In [11]:
tmp_citiesframe =  ((spark
                        .read
                        .format('jdbc')
                        .option('url', mssqlsource)
                        .option('driver', mssqldriver)
                        .option('user', mssqluser)
                        .option('password', mssqlpwd)
                        .option('query', 
                                """ 
                                    select 

                                        apc.cityid,
                                        apc.cityname,
                                        apc.stateprovinceid,
                                        apc.latestrecordedpopulation

                                    from application.cities as apc

                                    left join

                                        application.cities_archive as apca
                                    on apc.cityid = apca.cityid
                                    
                                    """) 
                          )).load()


In [12]:
tmp_stateframe = ((spark
                    .read
                    .format('jdbc')
                    .option('url', mssqlsource)
                    .option('driver', mssqldriver)
                    .option('user', mssqluser)
                    .option('password', mssqlpwd)
                    .option('query',
                            """ 
                                select

                                    asp.countryid,
                                    asp.stateprovinceid,
                                    asp.stateprovincename,
                                    asp.salesterritory,
                                    asp.stateprovincecode,
                                    asp.latestrecordedpopulation
                                    
                                from application.stateprovinces as asp
                                
                                left join
                                    application.stateprovinces_archive as aspa
                                    
                                on asp.stateprovinceid = aspa.stateprovinceid
                            
                            """) 
                   )).load()

In [13]:
#joining dataframes based on stateprovinceid column
state_cities = (( 
                 tmp_stateframe
                    .join(tmp_citiesframe,
                          on= 'stateprovinceid',
                          how= 'left')
                    
                    #select and rename columns
                    .select(tmp_stateframe.countryid,
                            tmp_stateframe.stateprovinceid,
                            tmp_stateframe.stateprovincename,
                            tmp_stateframe.salesterritory,
                            tmp_stateframe.stateprovincecode,
                            tmp_citiesframe.cityid,
                            tmp_citiesframe.cityname,
                            tmp_stateframe.latestrecordedpopulation.alias('statepopulation'),
                            
                            #removing null values
                            f.coalesce(tmp_citiesframe.latestrecordedpopulation, f.lit(0)).alias('citypopulation'))
                 ))


In [14]:
#rename column
statecities_final = (( 
                      state_cities
                        .withColumnRenamed('stateprovinceid', 'stateid')
                        .withColumnRenamed('stateprovincename', 'statename')

                      ))



In [15]:
((
    statecities_final
        .write
            .format('jdbc')
            .option('url', f'{mssql_dwdest}')
            .option('driver', f'{mssqldriver}')
            .option('user', f'{mssqluser}')
            .option('password', f'{mssqlpwd}')
            .option('dbtable', 'dimension.country')
        .save(mode= 'Overwrite')
    
))